# Flood Area Segmentation

This notebook reproduces the project in Google Colab. It must run from a copy of the supplied `ML_Lab_Teacher_Student_Model` folder. The data download cell requires a Kaggle API token for the assigned dataset.

## 0. Colab setup

Upload the complete project folder to Google Drive, or set `REPO_URL` if you have pushed the folder to a private/public Git repository. Then run the next cell once.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = ''  # Optional: paste your Git repository URL here
PROJECT_DIR = Path('/content/ML_Lab_Teacher_Student_Model')

if IN_COLAB and REPO_URL and not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
if not PROJECT_DIR.exists():
    # When opened from the repository locally, use the current directory.
    PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'configs' / 'experiment.json').is_file():
    raise RuntimeError('Place this notebook inside ML_Lab_Teacher_Student_Model, upload that folder to /content, or set REPO_URL.')
os.chdir(PROJECT_DIR)
print('Project directory:', PROJECT_DIR)


In [ ]:
# Install the exact project dependencies.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 1. Dataset and audit (C1)

For Colab, upload `kaggle.json` only when the Kaggle downloader requests it. The project verifies the expected 290 image-mask pairs before use.

In [ ]:
raw = PROJECT_DIR / 'data' / 'raw'
if not ((raw / 'Image').is_dir() and (raw / 'Mask').is_dir()):
    print('Kaggle credentials may be required for the next command.')
    subprocess.run([sys.executable, 'scripts/download_data.py'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([sys.executable, '-m', 'src.run', '--stage', 'audit'], check=True)
RESULTS = PROJECT_DIR / 'results'
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'
audit = json.loads((RESULTS / 'checkpoints' / 'CP01_dataset_audit.json').read_text())
assert {k: audit['structure'][k] for k in ('labelled', 'unlabelled', 'validation', 'test')} == {'labelled':40, 'unlabelled':163, 'validation':29, 'test':58}
print('C1 passed')


In [ ]:
from IPython.display import Image, display
import pandas as pd
display(Image(filename=str(FIGURES / 'audit_class_balance.png')))
display(pd.read_csv(TABLES / 'cross_partition_duplicate_images.csv'))


## 2. Feature and capacity checks (C2)

The student has six direct RGB and HSV features. The teacher has 20 handcrafted features. The feature budget and allowed model families are enforced in source code and unit tests.

In [ ]:
features = pd.read_csv(TABLES / 'feature_table.csv')
schema = json.loads((TABLES / 'feature_schema.json').read_text())
assert schema['student']['dimension'] == 6
assert schema['teacher']['dimension'] == 20
display(features)
display(Image(filename=str(FIGURES / 'pipeline_architecture.png')))


## 3. Development selection (C3)

This stage uses only L masks, U images, and validation masks. It selects teacher-student configurations, thresholds, confidence tau values, and K-Means variants. It may take several minutes.

In [ ]:
subprocess.run([sys.executable, '-m', 'src.run', '--stage', 'development'], check=True)
frozen = json.loads((RESULTS / 'frozen_selection.json').read_text())
assert frozen['status'] == 'FROZEN_DEVELOPMENT_SELECTION'
print('C3 passed:', frozen['selection'])
display(pd.read_csv(TABLES / 'validation_pathway_comparison.csv'))
display(Image(filename=str(FIGURES / 'tau_study.png')))


## 4. Frozen final test (C4)

Run this only after C3 succeeds. It checks the frozen selection before reading test masks; do not use the final result to change a model setting.

In [ ]:
subprocess.run([sys.executable, '-m', 'src.run', '--stage', 'final'], check=True)
cp04 = json.loads((RESULTS / 'checkpoints' / 'CP04_final_test.json').read_text())
assert cp04['status'] == 'PASSED'
assert cp04['evidence']['test_pixels'] == 58 * 256 * 256
print('C4 passed over', cp04['evidence']['test_pixels'], 'pixels')
for name in ['supervised_teachers_test.csv', 'supervised_baselines_test.csv', 'hard_distillation_iou_matrix.csv', 'kmeans_test_results.csv', 'track_summary.csv']:
    print('\n', name)
    display(pd.read_csv(TABLES / name))


In [ ]:
for file in sorted(FIGURES.glob('qualitative_*.png')):
    display(Image(filename=str(file)))


## 5. Final checks

The final assertions verify the expected structure of generated output artifacts.

In [ ]:
import csv
def rows(name):
    with (TABLES / name).open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))
assert len(rows('hard_distillation_test_long.csv')) == 9
assert len(rows('hard_distillation_iou_matrix.csv')) == 3
assert len(rows('kmeans_test_results.csv')) == 5
assert len(rows('track_summary.csv')) == 2
assert len(rows('selected_model_error_analysis.csv')) == 58 * 4
print('C5 passed: tables, figures, and error-analysis records are present.')


## Conclusions and limitations

The selected hard RF to RF student reaches 75.76% F1 and 60.98% IoU, improving compact RF by 2.89 IoU points. The prescribed split has seven cross-partition exact duplicate groups, including an L-to-test pair, so the results describe this benchmark and should not be treated as deployment evidence.